<a href="https://colab.research.google.com/github/Joan-Njoki-Mwangi/Hospital_Management/blob/main/Hospital_management_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Objective**
* To load, clean, and integrate multiple datasets in order to create a comprehensive and analysis-ready data model that supports meaningful data visualisation and insight generation in Looker.

**Specific Objectives**

* *Data Loading and Integration*
> Import the relevant datasets and merge them appropriately to enrich individual tables with complementary information, ensuring relational consistency and eliminating data silos.

* *Data Cleaning and Transformation*
> Convert columns to their appropriate data types (e.g., dates(considered as objects format to datetime format) to ensure analytical accuracy and compatibility with Tableau.

* *Feature Engineering (Creating New Columns)*
> Derive new variables that enhance analytical depth, such as calculated metrics, demographic indicators, time-based attributes (e.g., year, month), and behavioural classifications that support segmentation and trend analysis.

* *Data Validation and Quality Assurance*
> Identify and handle missing values, duplicates, inconsistencies, and possible outliers to improve data reliability and ensure trustworthy visual outputs.

* *Preparation for Visualization*
> Structure the final dataset to align with key business questions, enabling effective dashboards that highlight trends, patterns, performance indicators, and relationships across patient, billing, appointment, and treatment data.

# **Installing dependencies & loading data**

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:


class DataLoader:

    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None

    def _detect_file_type(self):
        _, ext = os.path.splitext(self.file_path.lower())
        if ext == ".csv":
            return "csv"
        elif ext in [".xlsx", ".xls"]:
            return "excel"
        else:
            raise ValueError("Unsupported file type.")

    def _load_file(self):
        """Load CSV or Excel into self.df"""
        if self._detect_file_type() == "csv":
            self.df = pd.read_csv(self.file_path)
        else:
            self.df = pd.read_excel(self.file_path)

    def _validate_column(self, column_name):
        if column_name not in self.df.columns:
            raise ValueError(f"Column '{column_name}' not found in dataset.")

    def run(
        self,
        index_column=None,
        date_column=None,  # only one column now
        month_name=False,
        add_year_month=False
    ):
        # Load the file
        self._load_file()

        # Process date column if specified
        if date_column:
            self._validate_column(date_column)

            converted = pd.to_datetime(
                self.df[date_column],
                errors="coerce"        # invalid dates become NaT
            ).dt.normalize()              # remove timestamp

            self.df[date_column] = converted

            # Create Year and Month columns if needed
            if add_year_month:
                self.df[f"{date_column}_Year"] = converted.dt.year
                self.df[f"{date_column}_Year_month"] = converted.dt.strftime("%Y-%m")

            if month_name:
                self.df[f"{date_column}_Month"] = converted.dt.month_name()
            else:
                self.df[f"{date_column}_Month"] = converted.dt.month



        return self.df

In [4]:
# loading appointments data
loader = DataLoader("/content/appointments.csv")
appointments = loader.run(
    date_column = "appointment_date",
    month_name = True,
    add_year_month = True
)

print(appointments.sample(5))


    appointment_id patient_id doctor_id appointment_date appointment_time  \
105           A106       P037      D005       2023-10-29         11:15:00   
30            A031       P026      D006       2023-04-04         10:30:00   
49            A050       P045      D008       2023-08-16         15:00:00   
149           A150       P047      D003       2023-08-16         10:45:00   
65            A066       P033      D009       2023-05-10         11:45:00   

      Appointment type     status  Unnamed: 7  Unnamed: 8  \
105            Therapy  Scheduled         NaN         NaN   
30             Checkup  Completed         NaN         NaN   
49        Consultation    No-show         NaN         NaN   
149            Therapy  Completed         NaN         NaN   
65        Consultation    No-show         NaN         NaN   

     appointment_date_Year appointment_date_Year_month appointment_date_Month  
105                   2023                     2023-10                October  
30        

In [5]:
#loading billing data
loader = DataLoader("/content/billing.csv")
billing = loader.run(
    date_column = "bill_date",
    month_name = True,
    add_year_month = True
)

print(billing.sample(5))

print(billing.bill_date.max())


    bill_id patient_id treatment_id  bill_date   amount payment_method  \
114    B115       P049         T115 2023-10-25  4809.31      Insurance   
119    B120       P032         T120 2023-12-08   935.04      Insurance   
28     B029       P016         T029 2023-06-25  3565.03      Insurance   
158    B159       P016         T159 2023-04-08  4687.68    Credit Card   
62     B063       P050         T063 2023-06-29  1256.06      Insurance   

    payment_status  bill_date_Year bill_date_Year_month bill_date_Month  
114           Paid            2023              2023-10         October  
119           Paid            2023              2023-12        December  
28            Paid            2023              2023-06            June  
158        Pending            2023              2023-04           April  
62          Failed            2023              2023-06            June  
2023-12-30 00:00:00


In [6]:
#loading doctors data
loader = DataLoader("/content/doctors.csv")
doctors_info = loader.run()

doctors_info['doctors_name'] = doctors_info['first_name'] + ' ' + doctors_info['last_name']

doctors_info.drop(['first_name', 'last_name'], axis=1, inplace=True)


print(doctors_info.sample(5))

  doctor_id specialization  phone_number  years_experience   hospital_branch  \
2      D003     Pediatrics    8737740598                19   Eastside Clinic   
1      D002     Pediatrics    9004382050                24   Eastside Clinic   
5      D006     Pediatrics    6570137231                23  Central Hospital   
7      D008    Dermatology    9069162601                 5   Westside Clinic   
3      D004     Pediatrics    6594221991                28  Central Hospital   

                         email doctors_name  
2   dr.jane.smith@hospital.com   Jane Smith  
1   dr.jane.davis@hospital.com   Jane Davis  
5   dr.alex.davis@hospital.com   Alex Davis  
7  dr.linda.brown@hospital.com  Linda Brown  
3  dr.david.jones@hospital.com  David Jones  


In [7]:
#loading patients data
loader = DataLoader("/content/patients.csv")
patients_info = loader.run(
    date_column = "registration_date",
    month_name = True,
    add_year_month = True
)

patients_info['patients_name'] = patients_info['first_name'] + ' ' + patients_info['last_name']

patients_info.drop(['first_name', 'last_name'], axis=1, inplace=True)
patients_info['date_of_birth'] = pd.to_datetime(patients_info['date_of_birth'])

patients_info.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 13 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   patient_id                    50 non-null     object        
 1   gender                        50 non-null     object        
 2   date_of_birth                 50 non-null     datetime64[ns]
 3   contact_number                50 non-null     int64         
 4   address                       50 non-null     object        
 5   registration_date             50 non-null     datetime64[ns]
 6   insurance_provider            50 non-null     object        
 7   insurance_number              50 non-null     object        
 8   email                         50 non-null     object        
 9   registration_date_Year        50 non-null     int32         
 10  registration_date_Year_month  50 non-null     object        
 11  registration_date_Month       50 n

In [8]:
#loading treatment data
loader = DataLoader("/content/treatments.csv")
treatments = loader.run(
    index_column = "treatment_id",
    date_column = "treatment_date",
    month_name = True,
    add_year_month = True
)

print(treatments.sample(5))


    treatment_id appointment_id treatment_type         description     cost  \
43          T044           A044            MRI     Basic screening  4186.35   
4           T005           A005            ECG  Standard procedure   582.05   
58          T059           A059            ECG  Standard procedure   929.91   
139         T140           A140  Physiotherapy  Standard procedure  4019.13   
5           T006           A006   Chemotherapy  Standard procedure  1381.00   

    treatment_date  treatment_date_Year treatment_date_Year_month  \
43      2023-09-20                 2023                   2023-09   
4       2023-09-02                 2023                   2023-09   
58      2023-03-09                 2023                   2023-03   
139     2023-02-05                 2023                   2023-02   
5       2023-06-19                 2023                   2023-06   

    treatment_date_Month  
43             September  
4              September  
58                 March  
13

# **Data Checks**

In [9]:
"""
Print the maximum date for each dataframe's date column.

Parameters:
dfs: list of pandas dataframe
date_columns: list of column names(string) corresponding to each dataframe
names: list of names(string) corresponding to each dataframe
"""
def print_max_dates(dfs,date_columns,names = None):
  if names is None:
    names = [f"{i+1}." for i in range(len(dfs))]
  for df,col,name in zip(dfs,date_columns,names):
    if col not in df.columns:
      print(f"{name} has no {col} column")
      continue
    max_date = df[col].max()
    print(f"{name} max date in '{col}':{max_date} ")



In [10]:
# printing the latest dates for each dataset
data = [appointments, billing,patients_info,treatments]
date_columns = ["appointment_date","bill_date","registration_date","treatment_date"]

print_max_dates(data,date_columns)


1. max date in 'appointment_date':2023-12-30 00:00:00 
2. max date in 'bill_date':2023-12-30 00:00:00 
3. max date in 'registration_date':2023-12-12 00:00:00 
4. max date in 'treatment_date':2023-12-30 00:00:00 


*the latest record accross the 5 datasets is Dec 30th 2023*





# **Adding new fields**



* Age relative to the date of appointment
> To determine each patient’s age, the most appropriate approach is to calculate the time difference between their Date of Birth and their Appointment date. This calculation enables analysis of whether age influences key behavioural and financial patterns, such as late, early, or defaulted payments; frequency of visits (distinguishing frequent attendees from no-shows); preferred treatment types or departments; and the progression of patients from appointments to treatments.

* And, Age relative to the latest registration to determine general demographic traits of patients
>Creating another age column to reflect the overall demographic profile of patient records against the most recent registration date.




In [11]:
# Step 1: merging the two datasets (appointments,patients_info)
patient_appointment_records = pd.merge(appointments, patients_info[['date_of_birth','patients_name', 'gender','registration_date','insurance_number',
'address','insurance_provider','contact_number','email','date_of_birth','patient_id','gender']],how="left", on = "patient_id")
patient_appointment_records = patient_appointment_records.reset_index(drop=True)
print("Sum of null values: ", patient_appointment_records.columns.isnull().sum())
print( patient_appointment_records.info())

Sum of null values:  0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 23 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   appointment_id               200 non-null    object        
 1   patient_id                   200 non-null    object        
 2   doctor_id                    200 non-null    object        
 3   appointment_date             200 non-null    datetime64[ns]
 4   appointment_time             200 non-null    object        
 5     Appointment type           200 non-null    object        
 6   status                       200 non-null    object        
 7   Unnamed: 7                   0 non-null      float64       
 8   Unnamed: 8                   1 non-null      float64       
 9   appointment_date_Year        200 non-null    int32         
 10  appointment_date_Year_month  200 non-null    object        
 11  appointment_date_Month

In [12]:
patient_appointment_records.columns

Index(['appointment_id', 'patient_id', 'doctor_id', 'appointment_date',
       'appointment_time', '  Appointment type', 'status', 'Unnamed: 7',
       'Unnamed: 8', 'appointment_date_Year', 'appointment_date_Year_month',
       'appointment_date_Month', 'date_of_birth', 'patients_name', 'gender',
       'registration_date', 'insurance_number', 'address',
       'insurance_provider', 'contact_number', 'email', 'date_of_birth',
       'gender'],
      dtype='object')

In [13]:
#Step 2: finding the time difference between date of birth and date of appointment

  #step i) first by changing date of birth to datetime
patient_appointment_records.index.duplicated().sum()

patient_appointment_records = patient_appointment_records.loc[:,~patient_appointment_records.columns.duplicated()]
patient_appointment_records['age_at_appointment'] = (
   (patient_appointment_records['appointment_date'] -
    patient_appointment_records['date_of_birth'])
    .dt.days // 365)
patient_appointment_records[['patients_name','age_at_appointment']].sample(5)

,patients_name,age_at_appointment
179,Alex Johnson,33
126,David Wilson,30
77,Laura Johnson,33
112,Michael Wilson,25
125,Robert Wilson,57


In [14]:
#Step 3:Finding the time difference between date of birth and registration date
patient_appointment_records['age_at_registration'] = (
   (patient_appointment_records['registration_date'] -
    patient_appointment_records['date_of_birth'])
    .dt.days // 365)
patient_appointment_records[['patients_name','age_at_registration','age_at_appointment']].sample(5)

,patients_name,age_at_registration,age_at_appointment
68,Laura Davis,31,31
89,John Taylor,17,19
161,Jane Smith,67,68
3,Robert Wilson,55,57
159,Jane Wilson,70,73


In [15]:
patient_appointment_records.columns

Index(['appointment_id', 'patient_id', 'doctor_id', 'appointment_date',
       'appointment_time', '  Appointment type', 'status', 'Unnamed: 7',
       'Unnamed: 8', 'appointment_date_Year', 'appointment_date_Year_month',
       'appointment_date_Month', 'date_of_birth', 'patients_name', 'gender',
       'registration_date', 'insurance_number', 'address',
       'insurance_provider', 'contact_number', 'email', 'age_at_appointment',
       'age_at_registration'],
      dtype='object')

In [16]:
#examining the changes
patient_appointment_records[['patient_id','patients_name','date_of_birth','registration_date','appointment_date',
'age_at_registration','age_at_appointment']].sample(8)

,patient_id,patients_name,date_of_birth,registration_date,appointment_date,age_at_registration,age_at_appointment
172,P047,Jane Moore,1995-12-13,2022-05-20,2023-06-04,26,27
153,P012,Laura Davis,1991-12-08,2023-04-27,2023-03-06,31,31
179,P007,Alex Johnson,1989-06-08,2021-12-25,2023-01-07,32,33
15,P016,Michael Taylor,2000-07-22,2021-07-23,2023-06-30,21,22
22,P047,Jane Moore,1995-12-13,2022-05-20,2023-05-09,26,27
99,P029,David Smith,2005-05-15,2023-04-19,2023-03-02,17,17
11,P029,David Smith,2005-05-15,2023-04-19,2023-05-07,17,17
132,P048,Emily Miller,1983-03-24,2023-06-19,2023-03-23,40,40


In [ ]:
# # Step 1: merging the two datasets (appointments,patients_info)
# patient_appointment_records = pd.merge(patient_appointment_records, doctors_info[['doctors_name',
#                                                                                   'specialization', 'hospital_branch','doctor_id']],
#                                        how="left", on = "doctor_id")
# patient_appointment_records = patient_appointment_records.reset_index(drop=True)
# patient_appointment_records.info()

In [ ]:
# patient_appointment_records = pd.merge(patient_appointment_records, doctors_info[['doctors_name',
#                                                                                   'specialization', 'hospital_branch','doctor_id']],
#                                        how="left", on = "doctor_id")

In [17]:
import sqlite3

conn = sqlite3.connect(":memory:")

patient_appointment_records.to_sql("patients", conn, index=False)
doctors_info.to_sql("doctors", conn, index=False)
treatments.to_sql("treatments", conn, index=False)
billing.to_sql("billing", conn, index=False)

query = """
SELECT
p.patient_id,
d.doctor_id,
appointment_date_Year,
appointment_date_Year_month,
appointment_date_Month,
strftime('%W', appointment_date) as Week_Num,
appointment_date,
date_of_birth,
registration_date,
age_at_appointment,
age_at_registration,
patients_name,
gender,
insurance_number,
address,
insurance_provider,
contact_number,
p.email,
doctors_name,
specialization as department,
hospital_branch as hospital,
amount as billed_amount,
payment_status,
t.treatment_id as treatment_id_for_billing,
p.appointment_id,
p.status

FROM patients p
LEFT JOIN doctors d
ON p.doctor_id = d.doctor_id
LEFT JOIN treatments t
ON p.appointment_id = t.appointment_id and p.appointment_date = t.treatment_date
LEFT JOIN billing b
ON t.treatment_id = b.treatment_id and t.treatment_date = b.bill_date

"""

patient_appointment_records = pd.read_sql(query, conn)


In [18]:
# removing duplicates created while merging
patient_appointment_records = patient_appointment_records.loc[:,~patient_appointment_records.columns.duplicated()]
# checking on changes
patient_appointment_records.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 26 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   patient_id                   200 non-null    object 
 1   doctor_id                    200 non-null    object 
 2   appointment_date_Year        200 non-null    int64  
 3   appointment_date_Year_month  200 non-null    object 
 4   appointment_date_Month       200 non-null    object 
 5   Week_Num                     200 non-null    object 
 6   appointment_date             200 non-null    object 
 7   date_of_birth                200 non-null    object 
 8   registration_date            200 non-null    object 
 9   age_at_appointment           200 non-null    int64  
 10  age_at_registration          200 non-null    int64  
 11  patients_name                200 non-null    object 
 12  gender                       200 non-null    object 
 13  insurance_number    

In [19]:
#exporting to excel
patient_appointment_records.to_csv("patient_appointment_records.csv")